### RAG Knowledge Retrieval

Goal:
Build a retrieval system that finds relevant company policy context for customer support tickets.

In [21]:
import os
import pandas as pd

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [22]:
POLICY_DIR = "../docs/company_policies"
VECTORSTORE_DIR = "../vectorstore/faiss_policy_index"
REPORT_PATH = "../reports/rag_test_results.csv"

In [23]:
policy_files = os.listdir(POLICY_DIR)
policy_files

['account_policy.txt',
 'refund_policy.txt',
 'shipping_policy.txt',
 'technical_support_policy.txt',
 'warranty_policy.txt']

In [24]:
loader = DirectoryLoader(
    POLICY_DIR,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

documents = loader.load()

print("Number of documents loaded:", len(documents))

Number of documents loaded: 5


In [25]:
documents[0]

Document(metadata={'source': '..\\docs\\company_policies\\account_policy.txt'}, page_content='Account Policy\n\nCustomers can request help for account login, password reset, profile update, email change, and account recovery.\n\nFor login issues:\n- Verify the registered email address.\n- Ask the customer to reset their password.\n- Check whether the account is locked.\n- Escalate if the account may be compromised.\n\nFor email change requests, support agents must verify customer identity.\n\nFor security issues, support agents should not share sensitive account information without verification.\n\nAccount issues should be assigned to Account Support.\nCompromised account reports should be marked as high priority.')

In [26]:
for doc in documents:
    print("=" * 80)
    print("Source:", doc.metadata["source"])
    print(doc.page_content[:500])

Source: ..\docs\company_policies\account_policy.txt
Account Policy

Customers can request help for account login, password reset, profile update, email change, and account recovery.

For login issues:
- Verify the registered email address.
- Ask the customer to reset their password.
- Check whether the account is locked.
- Escalate if the account may be compromised.

For email change requests, support agents must verify customer identity.

For security issues, support agents should not share sensitive account information without verification.

Ac
Source: ..\docs\company_policies\refund_policy.txt
Refund Policy

Customers can request a refund within 30 days of purchase.

Refunds are allowed when:
- The product is damaged on arrival.
- The wrong item was delivered.
- The product does not match the description.
- The customer was charged incorrectly.
- The customer received a defective item.

Refunds are not allowed when:
- The product was damaged because of customer misuse.
- The refund 

In [27]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks created:", len(chunks))

Number of chunks created: 10


In [28]:
for i, chunk in enumerate(chunks[:5]):
    print("=" * 80)
    print("Chunk:", i + 1)
    print("Source:", chunk.metadata["source"])
    print(chunk.page_content)

Chunk: 1
Source: ..\docs\company_policies\account_policy.txt
Account Policy

Customers can request help for account login, password reset, profile update, email change, and account recovery.

For login issues:
- Verify the registered email address.
- Ask the customer to reset their password.
- Check whether the account is locked.
- Escalate if the account may be compromised.

For email change requests, support agents must verify customer identity.

For security issues, support agents should not share sensitive account information without verification.
Chunk: 2
Source: ..\docs\company_policies\account_policy.txt
Account issues should be assigned to Account Support.
Compromised account reports should be marked as high priority.
Chunk: 3
Source: ..\docs\company_policies\refund_policy.txt
Refund Policy

Customers can request a refund within 30 days of purchase.

Refunds are allowed when:
- The product is damaged on arrival.
- The wrong item was delivered.
- The product does not match the d

In [29]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [30]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("FAISS vectorstore created successfully.")

FAISS vectorstore created successfully.


In [31]:
query = "My product arrived damaged and I want a refund."

results = vectorstore.similarity_search(query, k=2)

for i, doc in enumerate(results, 1):
    print("=" * 80)
    print(f"Result {i}")
    print("Source:", doc.metadata["source"])
    print(doc.page_content)

Result 1
Source: ..\docs\company_policies\refund_policy.txt
Refunds are not allowed when:
- The product was damaged because of customer misuse.
- The refund request is made after 30 days.
- The customer cannot provide proof of purchase.
- The item was purchased from an unauthorized seller.

For damaged products, customers may choose either a refund or replacement.

Refund requests should be assigned to the Billing or Customer Support queue depending on the issue.
High-value refund disputes should be marked as high priority.
Result 2
Source: ..\docs\company_policies\refund_policy.txt
Refund Policy

Customers can request a refund within 30 days of purchase.

Refunds are allowed when:
- The product is damaged on arrival.
- The wrong item was delivered.
- The product does not match the description.
- The customer was charged incorrectly.
- The customer received a defective item.


In [32]:
test_queries = [
    "My product arrived damaged and I want a refund.",
    "My order has not arrived and tracking is not updating.",
    "My laptop battery drains very quickly.",
    "I cannot login to my account.",
    "My device stopped working after 6 months.",
    "The package says delivered but I did not receive it.",
    "I forgot my password and cannot access my account.",
    "The charger stopped working within warranty period."
]

for query in test_queries:
    results = vectorstore.similarity_search(query, k=1)

    print("=" * 100)
    print("Ticket:", query)
    print("Retrieved Source:", results[0].metadata["source"])
    print("Retrieved Text:")
    print(results[0].page_content[:500])

Ticket: My product arrived damaged and I want a refund.
Retrieved Source: ..\docs\company_policies\refund_policy.txt
Retrieved Text:
Refunds are not allowed when:
- The product was damaged because of customer misuse.
- The refund request is made after 30 days.
- The customer cannot provide proof of purchase.
- The item was purchased from an unauthorized seller.

For damaged products, customers may choose either a refund or replacement.

Refund requests should be assigned to the Billing or Customer Support queue depending on the issue.
High-value refund disputes should be marked as high priority.
Ticket: My order has not arrived and tracking is not updating.
Retrieved Source: ..\docs\company_policies\shipping_policy.txt
Retrieved Text:
Shipping Policy

Standard delivery usually takes 5 to 7 business days.

Customers can contact support when:
- The order is delayed.
- Tracking information is not updating.
- The package is marked as delivered but not received.
- The wrong delivery address

In [33]:
test_cases = [
    {
        "ticket": "My product arrived damaged and I want a refund.",
        "expected_policy": "refund_policy.txt"
    },
    {
        "ticket": "My order has not arrived and tracking is not updating.",
        "expected_policy": "shipping_policy.txt"
    },
    {
        "ticket": "My laptop battery drains very quickly.",
        "expected_policy": "technical_support_policy.txt"
    },
    {
        "ticket": "I cannot login to my account.",
        "expected_policy": "account_policy.txt"
    },
    {
        "ticket": "My device stopped working after 6 months.",
        "expected_policy": "warranty_policy.txt"
    },
    {
        "ticket": "The package says delivered but I did not receive it.",
        "expected_policy": "shipping_policy.txt"
    },
    {
        "ticket": "I forgot my password and cannot access my account.",
        "expected_policy": "account_policy.txt"
    },
    {
        "ticket": "The charger stopped working within warranty period.",
        "expected_policy": "warranty_policy.txt"
    }
]

rag_results = []

for case in test_cases:
    ticket = case["ticket"]
    expected_policy = case["expected_policy"]

    retrieved_docs = vectorstore.similarity_search(ticket, k=1)

    retrieved_source = retrieved_docs[0].metadata["source"]
    retrieved_policy = os.path.basename(retrieved_source)

    is_correct = expected_policy == retrieved_policy

    rag_results.append({
        "ticket": ticket,
        "expected_policy": expected_policy,
        "retrieved_policy": retrieved_policy,
        "is_correct": is_correct,
        "retrieved_text": retrieved_docs[0].page_content
    })

rag_results_df = pd.DataFrame(rag_results)
rag_results_df

,ticket,expected_policy,retrieved_policy,is_correct,retrieved_text
0,My product arrived damaged and I want a refund.,refund_policy.txt,refund_policy.txt,True,Refunds are not allowed when:\n- The product w...
1,My order has not arrived and tracking is not u...,shipping_policy.txt,shipping_policy.txt,True,Shipping Policy\n\nStandard delivery usually t...
2,My laptop battery drains very quickly.,technical_support_policy.txt,technical_support_policy.txt,True,Technical Support Policy\n\nTechnical support ...
3,I cannot login to my account.,account_policy.txt,account_policy.txt,True,Account Policy\n\nCustomers can request help f...
4,My device stopped working after 6 months.,warranty_policy.txt,technical_support_policy.txt,False,Technical Support Policy\n\nTechnical support ...
5,The package says delivered but I did not recei...,shipping_policy.txt,shipping_policy.txt,True,Shipping Policy\n\nStandard delivery usually t...
6,I forgot my password and cannot access my acco...,account_policy.txt,account_policy.txt,True,Account Policy\n\nCustomers can request help f...
7,The charger stopped working within warranty pe...,warranty_policy.txt,warranty_policy.txt,True,Warranty Policy\n\nProducts include a 1-year l...


In [34]:
accuracy = rag_results_df["is_correct"].mean()

print("RAG Retrieval Accuracy:", accuracy)

RAG Retrieval Accuracy: 0.875


In [35]:
rag_results_df.to_csv(REPORT_PATH, index=False)

print("Saved report to:", REPORT_PATH)

Saved report to: ../reports/rag_test_results.csv


In [36]:
vectorstore.save_local(VECTORSTORE_DIR)

print("Vectorstore saved at:", VECTORSTORE_DIR)

Vectorstore saved at: ../vectorstore/faiss_policy_index


In [37]:
loaded_vectorstore = FAISS.load_local(
    VECTORSTORE_DIR,
    embeddings=embedding_model,
    allow_dangerous_deserialization=True
)

print("Vectorstore loaded successfully.")

Vectorstore loaded successfully.


In [38]:
query = "My account is locked and I cannot login."

results = loaded_vectorstore.similarity_search(query, k=2)

for i, doc in enumerate(results, 1):
    print("=" * 80)
    print(f"Result {i}")
    print("Source:", doc.metadata["source"])
    print(doc.page_content)

Result 1
Source: ..\docs\company_policies\account_policy.txt
Account Policy

Customers can request help for account login, password reset, profile update, email change, and account recovery.

For login issues:
- Verify the registered email address.
- Ask the customer to reset their password.
- Check whether the account is locked.
- Escalate if the account may be compromised.

For email change requests, support agents must verify customer identity.

For security issues, support agents should not share sensitive account information without verification.
Result 2
Source: ..\docs\company_policies\account_policy.txt
Account issues should be assigned to Account Support.
Compromised account reports should be marked as high priority.
